# 03 — Normalized count, volume and time scores

Use the same first 200 AAPL trades and the agreed six vectorized differences. Build the nine-cell matrix for every exact price, then score and rank prices directly. No local peaks, neighboring-price tests, visits, smoothing or price bins. Earlier notebooks are preserved.

Support/resistance reward quick immediate reversals; continuation rewards a longer delay before the observed continuation. Overall occupancy stays separate. These are descriptive relative scores, not probabilities or validated trading signals.


In [ ]:
# Basic setup. Generated data and charts must stay outside the repository.
import os
import sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path("D:/TradingML/runtimes")
SOURCE_RUN = RUNTIME_ROOT / "level-book-v8/reaction-demo/20261020T204419563339Z"
EXPECTED_SOURCE_HASH = "b40270ba8fc4055d515fa9061ce5333cce152890de23e249dbbc827e7e0ec70d"
N_EVENTS = 200
# Explicit illustrative settings, fixed before inspecting the result.
TAU_SECONDS = 1.0          # Same 1-second reference used in our discussion; not a candle timeframe.
SCORE_THRESHOLD = 0.60     # A display/selection rule, not a calibrated confidence threshold.
WEIGHT_COUNT = 1.0 / 3     # Equal weights reproduce (normalized count + volume + time) / 3.
WEIGHT_VOLUME = 1.0 / 3
WEIGHT_TIME = 1.0 / 3
assert TAU_SECONDS > 0
assert 0 < SCORE_THRESHOLD <= 1
assert min(WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME) >= 0
assert np.isclose(WEIGHT_COUNT + WEIGHT_VOLUME + WEIGHT_TIME, 1.0)
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required root unavailable"
RUN = RUNTIME_ROOT / "level-book-v8/normalized-level-scores" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
print("Output directory:", RUN)

## 1. Load and verify the original sample
This reads the saved canonical-derived snapshot, not raw SIP flatfiles. We validate its original content hash before taking the first 200 rows. If it is missing or changed, stop rather than silently fetching a different sample.

The original extraction excluded delayed/unknown-clock and condition-ineligible trades. Its audit and provenance are retained. Price is stored in exact units of 1/10,000 dollar. Ordinal is metadata, not a predictor: order remains `(SIP timestamp, ordinal)`, exactly as in notebook 01. Time differences can be zero for equal timestamps.

We slice **before** calculating shifts. Event 201 must not reveal the departure of event 200.

In [ ]:
# Recreate the original loader's hashed records, in their original order.
source = pl.read_parquet(SOURCE_RUN / "source_trades.parquet")
provenance = json.loads((SOURCE_RUN / "source_provenance.json").read_text(encoding="utf-8"))
source_records = source.select("ordinal", "time_us", "price_units", "volume").to_dicts()
# ClickHouse JSON encoded integral sizes as integers; Parquet promoted the
# mixed size column to Float64. Restore only that JSON representation for hashing.
for record in source_records:
    if float(record["volume"]).is_integer():
        record["volume"] = int(record["volume"])
source_hash = hashlib.sha256(json.dumps(source_records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance["sample_sha256"] == EXPECTED_SOURCE_HASH
assert source.height == 2000 and provenance["ticker"] == "AAPL"
assert provenance["day"] == "2026-08-20"
assert 3 <= N_EVENTS <= source.height
assert source["ordinal"].n_unique() == source.height
assert source.select("time_us", "ordinal").equals(source.sort("time_us", "ordinal").select("time_us", "ordinal"))
raw = source.head(N_EVENTS)
# These are the three original quantities; event and ordinal identify rows.
display(raw.select("event", "ordinal", "time", "price", "volume").head(15))
print("Selected events:", raw.height, "unique prices:", raw["price_units"].n_unique())
print("Time span:", raw["time"][0], "to", raw["time"][-1])
display(pl.DataFrame(provenance["audit"]))

## 2. Our agreed step one: six vectorized differences
For any column x:

- backward = current minus previous;
- forward = next minus current.

Time differences are seconds, price differences dollars, and volume differences changes in individual trade size. Arrival and departure are the signs of the respective price differences. A zero price difference means unchanged, not missing.

The first backward values and final forward values remain null. Forward time is the current trade's completed hold contribution. We do not add elapsed time for the pending final row or pretend its reaction is unchanged.

In [ ]:
def step_one(frame):
    """Derive adjacent-row differences on precisely the supplied prefix."""
    return frame.with_columns(
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
step_columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
                "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(step_columns).head(15))
display(step1.select(step_columns).tail(3))

## 3. Group by exact price and the nine direction pairs

| Arrival / departure | Down (−1) | Unchanged (0) | Up (+1) |
|---|---|---|---|
| Down (−1) | Down continuation | Pause after decline | Support-like bounce |
| Unchanged (0) | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up (+1) | Resistance-like rejection | Pause after rise | Up continuation |

Each trade belongs to one cell when both directions are known. There are no visit IDs. For example, down→unchanged followed by unchanged→up is **not** merged into one bounce.

Each cell stores count, current traded volume, and sums of all six differences. Dividing a difference sum by count gives its cell mean. Signed volume changes can cancel, so they remain explanatory measurements, not an automatic strength score.

Boundary rows are displayed separately and still contribute to the total activity distribution.

In [ ]:
# These are the six differences already calculated in step one.
# Listing their names lets us apply the same sum operation to each column.
difference_columns = ["time_backward", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"]
# Keep rows with BOTH directions known so each can be assigned to one of nine groups.
# pl.col selects a column, is_not_null checks availability, and & requires both checks.
# A direction of zero (unchanged) is valid and is retained.
complete = step1.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
# Retain rows missing either direction in a separate audit table; | means OR.
# The first arrival and final departure are unknown, not unchanged.
# Keeping them separately prevents silent loss of their count and volume.
boundary_rows = step1.filter(pl.col("arrival").is_null() | pl.col("departure").is_null())
# Group by exact integer price AND arrival AND departure.
# For example, all rows at one price with arrival=-1 and departure=+1 form one group.
# Integer price units avoid floating-point equality problems. agg calculates one row per group.
profile = complete.group_by("price_units", "arrival", "departure").agg(
    # Count the trades in this group. alias gives the result its readable column name.
    # This is a trade count, not a count of independent visits.
    pl.len().alias("trade_count"),
    # Sum CURRENT trade sizes in this group, not next-trade size or volume change.
    # Thus bounce-group volume means volume on trades immediately followed by an upward move.
    pl.col("volume").sum().alias("volume_sum"),
    # Build one sum expression per difference column; * expands the list into agg arguments.
    # Forward-time sum is the group's completed hold contribution, measured in seconds.
    # Backward time is kept separately: adding the two time sums would double-count intervals.
    # Price and volume differences are signed; positive and negative values can cancel.
    # Dividing each sum by trade_count yields that group's mean difference.
    *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
# Close the aggregation, add a dollar-price column for display, then sort the rows.
# Grouping remains on price_units; dividing by 10,000 is presentation only.
# Sorting by price and both directions makes the table easier to inspect.
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units", "arrival", "departure")
# Display the first 18 aggregated rows only; the full profile remains in memory.
# This limits screen output, not the data used by subsequent calculations.
display(profile.head(18))
# Report how many original trade rows were classified and how many have unknown boundaries.
# DataFrame.height is its number of rows; these two counts should total N_EVENTS.
print("Classified rows:", complete.height, "boundary rows:", boundary_rows.height)
# Show the original event identities and known directions of the boundary rows.
# select chooses display columns; it does not modify or discard the saved boundary table.
display(boundary_rows.select("event", "price", "volume", "arrival", "departure"))

# Separately aggregate ALL selected trades by price, including the boundary rows.
# Activity totals need no next-trade outcome, so their count and volume are already known.
# This table describes overall trading concentration, not a particular reaction.
activity = raw.group_by("price_units").agg(pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume")).sort("price_units")
# Rank prices by total trade count descending; break ties by lower price first.
# Select the first price_units value to choose one reproducible example for the matrix.
# "Busiest" here means most trades, not greatest volume or strongest support.
busiest = activity.sort(["total_trades", "price_units"], descending=[True, False])["price_units"][0]
# Explicitly create all nine (arrival, departure) pairs: each direction is -1, 0 or +1.
# The departure list repeats three times, once for each arrival direction.
# Int8 matches the direction-column types in profile so the join uses compatible keys.
grid = pl.DataFrame({"arrival": [-1,-1,-1,0,0,0,1,1,1], "departure": [-1,0,1]*3}).cast(pl.Int8)
# Filter the profile to the chosen price, then match rows by both directions.
# A LEFT join preserves all nine grid rows, even if a combination never occurred.
# An unobserved combination has null aggregate values after this join.
matrix = grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
# Print the chosen price in dollars so the displayed matrix is clearly identified.
print("Nine cells at price:", busiest / 10000)
# Display directions plus count, volume and completed hold time for each of nine cells.
# Here null means NO MATCHING GROUP, so those three totals are correctly displayed as zero.
# This does not replace unknown directions or pending forward differences in the source.
# fill_null affects this display expression; the original matrix/profile remain unchanged.
display(matrix.select("arrival", "departure", pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0)))


## 4. Turn four directional cells into four scores per price
We retain all nine cells in the reaction profile. Four cells have the specific directional interpretations we agreed:

| Score | Arrival | Departure | Time component |
|---|---:|---:|---|
| Support | -1 | +1 | Quick: tau / (tau + mean delay) |
| Resistance | +1 | -1 | Quick: tau / (tau + mean delay) |
| Up continuation | +1 | +1 | Long: mean delay / (tau + mean delay) |
| Down continuation | -1 | -1 | Long: mean delay / (tau + mean delay) |

For each type separately, divide its count by the largest count across observed prices; do the same for volume. Mean delay = forward-time sum / count. Then take the weighted sum of normalized count, normalized volume, and the time component. Equal weights give the arithmetic average. All components and scores lie in [0,1]. If a cell has no observations, its score is zero; its mean delay remains unknown.

Example: normalized count=1, normalized volume=0.5 and mean delay=0.1 seconds with tau=1 gives a support score (1+0.5+0.9091)/3 = 0.8030. The same activity with a 4-second delay gives (1+0.5+0.2)/3 = 0.5667. Continuation reverses the preference for time.

We do NOT reinterpret unchanged cells as continuations. The nine-cell profile preserves them. A row's forward delay is only the interval to its next eligible trade, not the full time spent across several same-price trades. Zero elapsed time can occur because SIP timestamps tie; it gives quick=1 and long=0, not proof of a physically instantaneous response.

In [ ]:
# Define the four roles and their time interpretation explicitly.
roles = pl.DataFrame({
    "role": ["support", "resistance", "up_continuation", "down_continuation"],
    "arrival": [-1, 1, 1, -1],
    "departure": [1, -1, 1, -1],
    "time_preference": ["quick", "quick", "long", "long"],
}).with_columns(pl.col("arrival", "departure").cast(pl.Int8))

def score_prices(price_activity, reaction_profile, tau, weights):
    """Score every price/role, including empty cells. No neighbor or peak tests."""
    assert np.isfinite(tau) and tau > 0
    assert len(weights) == 3 and min(weights) >= 0 and np.isclose(sum(weights), 1)
    # Cross join creates all four roles at every price, even when a cell is absent.
    table = price_activity.join(roles, how="cross").join(
        reaction_profile.select("price_units", "arrival", "departure", "trade_count", "volume_sum", "time_forward_sum"),
        on=["price_units", "arrival", "departure"], how="left",
    ).with_columns(
        # Missing group means zero observations/totals, not an unknown source direction.
        pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0),
    ).with_columns(
        # over('role') computes each role's maximum across prices, without merging roles.
        pl.col("trade_count").max().over("role").alias("max_count"),
        pl.col("volume_sum").max().over("role").alias("max_volume"),
        # No observations means no known mean delay; leave it null rather than set it to zero.
        pl.when(pl.col("trade_count") > 0).then(pl.col("time_forward_sum") / pl.col("trade_count"))
          .otherwise(None).alias("mean_delay_seconds"),
    ).with_columns(
        # If a role has no evidence anywhere, its count/volume components are zero.
        pl.when(pl.col("max_count") > 0).then(pl.col("trade_count") / pl.col("max_count"))
          .otherwise(0.0).alias("count_component"),
        pl.when(pl.col("max_volume") > 0).then(pl.col("volume_sum") / pl.col("max_volume"))
          .otherwise(0.0).alias("volume_component"),
        # Select QUICK for reversal roles and LONG for continuation roles.
        pl.when(pl.col("trade_count") == 0).then(0.0)
          .when(pl.col("time_preference") == "quick").then(tau / (tau + pl.col("mean_delay_seconds")))
          .otherwise(pl.col("mean_delay_seconds") / (tau + pl.col("mean_delay_seconds"))).alias("time_component"),
    ).with_columns(
        # Expose the contribution of each term so the final score is easy to audit.
        (weights[0] * pl.col("count_component")).alias("count_contribution"),
        (weights[1] * pl.col("volume_component")).alias("volume_contribution"),
        (weights[2] * pl.col("time_component")).alias("time_contribution"),
        (pl.col("price_units") / 10000).alias("price"),
    ).with_columns(
        # An unobserved cell cannot earn a score from a time bonus.
        pl.when(pl.col("trade_count") > 0)
          .then(pl.col("count_contribution") + pl.col("volume_contribution") + pl.col("time_contribution"))
          .otherwise(0.0).alias("score"),
    )
    return table.sort(["role", "score", "price_units"], descending=[False, True, False])

weights = (WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME)
scores = score_prices(activity, profile, TAU_SECONDS, weights)
details = ["price", "role", "trade_count", "volume_sum", "time_forward_sum", "mean_delay_seconds",
           "count_component", "volume_component", "time_component", "score"]
display(scores.select(details).head(20))

## 5. Apply the threshold and sort
A price/role passes when score >= SCORE_THRESHOLD. There is no hidden minimum-count or peak requirement. Consequently a sparse price can score highly; the observation counts must stay visible. A price may pass for more than one role. Support and resistance tables identify reversal candidates; continuation is a separate behavior label, not automatically a support/resistance level.

Max-normalization is relative to this observed sample. Adding trades at other prices can change a price's normalized score even if that price has no new trades. A score of 0.8 does not mean an 80% probability of a bounce. Equal numerical thresholds across roles are a simple convention, not calibrated comparability.

This rule rewards reversal evidence without subtracting opposing continuations. We are deliberately implementing the agreed simple formula before adding that complexity.

In [ ]:
# Retain both passing and failing rows. The threshold flag is the entire selection rule.
scores = scores.with_columns((pl.col("score") >= SCORE_THRESHOLD).alias("passes_threshold"))
selected = scores.filter(pl.col("passes_threshold")).sort(["score", "price_units", "role"], descending=[True, False, False])
# Show each role separately so continuation is not confused with support/resistance.
for role in roles["role"].to_list():
    print("Role:", role, "threshold:", SCORE_THRESHOLD)
    display(scores.filter(pl.col("role") == role).select(*details, "passes_threshold").head(12))
print("All selected price/role pairs:")
display(selected.select(*details, "passes_threshold"))
# A compact table lists all four scores at each exact price, including zeros.
score_grid = scores.select("price", "role", "score").pivot(on="role", index="price", values="score").sort("price")
display(score_grid)
print("Selected pairs:", selected.height, "distinct selected prices:", selected["price_units"].n_unique())

## 6. Keep overall occupancy separate and explain one score
Overall completed occupancy is the sum of forward-time contributions at a price, including unchanged reactions and the first boundary row when its next trade is known. The final unresolved interval contributes no duration. It is not added to support/resistance scores.

A large overall occupancy can be many brief intervals or a few long ones. Reaction delay is the mean interval within one specific matrix cell. Neither measurement identifies executable liquidity or guarantees a future hold.

The sample covers only a few seconds. With the illustrative tau=1 second, most immediate delays may be tiny: quick-response components approach 1 and long-delay components approach 0. The table below shows this rather than silently choosing a better-looking tau. Changing tau is an explicit experiment.

In [ ]:
# Keep unobserved final duration out of the sum; count known intervals explicitly.
occupancy = step1.group_by("price_units").agg(
    pl.col("time_forward").sum().alias("completed_occupancy_seconds"),
    pl.col("time_forward").count().alias("known_intervals"),
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units")
display(occupancy)
display(scores.filter(pl.col("trade_count") > 0).group_by("role").agg(
    pl.col("mean_delay_seconds").min().alias("minimum_mean_delay"),
    pl.col("mean_delay_seconds").median().alias("median_mean_delay"),
    pl.col("mean_delay_seconds").max().alias("maximum_mean_delay"),
    pl.col("time_component").mean().alias("average_time_component"),
))
# Trace the highest-scoring observed support cell, whether or not it passes the threshold.
example = scores.filter((pl.col("role") == "support") & (pl.col("trade_count") > 0)).head(1)
if example.height:
    row = example.row(0, named=True)
    print("Support example at", row["price"], "using", row["trade_count"], "observed rows")
    display(example.select(*details, "count_contribution", "volume_contribution", "time_contribution"))
    print("Score =", row["count_contribution"], "+", row["volume_contribution"], "+", row["time_contribution"], "=", row["score"])
    # Display the exact source rows responsible for its count, volume and time sum.
    display(complete.filter((pl.col("price_units") == row["price_units"]) &
                            (pl.col("arrival") == -1) & (pl.col("departure") == 1))
            .select("event", "price", "volume", "arrival", "departure", "time_forward"))
else:
    print("No completed down-to-up observations; no support example can be shown.")

## 7. Visualize scores directly
Every observed price is shown, not just local peaks. Each panel is one reaction role; the dashed line is the same explicit threshold. These are scores calculated at the final sample cutoff, not signals backdated to earlier trades.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), constrained_layout=True, sharex=True, sharey=True)
for ax, role in zip(axes, roles["role"].to_list()):
    table = scores.filter(pl.col("role") == role).sort("price_units")
    passed = table.filter(pl.col("passes_threshold"))
    ax.plot(table["price"], table["score"], ".-", linewidth=0.8, label="all observed prices")
    ax.scatter(passed["price"], passed["score"], color="darkorange", zorder=3, label="passes threshold")
    ax.axhline(SCORE_THRESHOLD, color="red", linestyle="--", label="threshold")
    ax.set(title=role, ylabel="Normalized score", ylim=(-0.03, 1.03))
axes[0].legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Exact price")
fig.savefig(RUN / "normalized_scores.png", dpi=140)
plt.show()
plt.close(fig)

## 8. Verify the formula and save all evidence
Checks cover a hand-calculated score, the opposite effect of delay on reversals versus continuations, no-observation cells, zero-delay cells, normalization bounds, source accounting and prefix isolation. These check implementation, not predictive quality.

No database writes occur. Source snapshots, matrices, all scores (including failed candidates), selected rows and the chart go to the external runtime directory. Earlier notebooks and source samples remain unchanged.

In [ ]:
# Small synthetic profile: identical count/volume, different forward delays.
toy_activity = pl.DataFrame({"price_units": [100, 101, 102], "total_trades": [6,6,0], "total_volume": [60.,60.,0.]})
toy_profile = pl.DataFrame({"price_units": [100,101,100,101], "arrival": [-1,-1,1,1], "departure": [1,1,1,1],
                            "trade_count": [3,3,3,3], "volume_sum": [30.,30.,30.,30.], "time_forward_sum": [0.3,12.,0.3,12.]}).with_columns(pl.col("arrival","departure").cast(pl.Int8))
test = score_prices(toy_activity, toy_profile, 1.0, (1/3,1/3,1/3))
def value(price, role, column="score"):
    return test.filter((pl.col("price_units") == price) & (pl.col("role") == role))[column][0]
assert np.isclose(value(100,"support"), (1 + 1 + 1/1.1)/3)
assert value(100,"support") > value(101,"support")
assert value(100,"up_continuation") < value(101,"up_continuation")
assert test.filter(pl.col("price_units") == 102)["score"].sum() == 0
assert test.filter(pl.col("role") == "resistance")["score"].sum() == 0
zero_delay = score_prices(toy_activity, toy_profile.with_columns(pl.lit(0.).alias("time_forward_sum")), 1., weights)
assert zero_delay.filter((pl.col("price_units") == 100) & (pl.col("role") == "support"))["time_component"][0] == 1
assert zero_delay.filter((pl.col("price_units") == 100) & (pl.col("role") == "up_continuation"))["time_component"][0] == 0
assert np.isfinite(scores["score"].to_numpy()).all()
assert scores["score"].min() >= 0 and scores["score"].max() <= 1 + 1e-12
assert scores.height == activity.height * 4
assert scores.filter(pl.col("trade_count") == 0)["score"].sum() == 0
assert profile["trade_count"].sum() + boundary_rows.height == N_EVENTS
assert np.isclose(profile["volume_sum"].sum() + boundary_rows["volume"].sum(), raw["volume"].sum())
assert np.isclose(occupancy["completed_occupancy_seconds"].sum(), (raw["time_us"][-1] - raw["time_us"][0])/1e6)
assert step1["departure"][-1] is None
changed = source.with_columns(pl.when(pl.col("event") > N_EVENTS).then(pl.col("price_units") + 100000).otherwise(pl.col("price_units")).alias("price_units"))
assert step_one(changed.head(N_EVENTS)).equals(step1)
print("PASS: hand formula, delay direction, empty/zero-delay cells, score bounds, accounting, prefix isolation")

# Save both complete tables and selected rows so the threshold remains auditable.
raw.write_parquet(RUN / "selected_trades.parquet")
step1.write_parquet(RUN / "step_one.parquet")
profile.write_parquet(RUN / "nine_direction_profile.parquet")
scores.write_parquet(RUN / "all_price_role_scores.parquet")
selected.write_parquet(RUN / "selected_price_role_scores.parquet")
occupancy.write_parquet(RUN / "overall_occupancy.parquet")
NOTEBOOK = REPO / "research/level_book/v8/03_count_volume_time_scores.ipynb"
manifest = dict(source_run=str(SOURCE_RUN), source_sha256=source_hash, selected_events=N_EVENTS,
                tau_seconds=TAU_SECONDS, threshold=SCORE_THRESHOLD, weights=list(weights),
                unique_prices=activity.height, scored_pairs=scores.height, selected_pairs=selected.height,
                selected_by_role=selected.group_by("role").len().sort("role").to_dicts(),
                checks="passed", notebook_sha256=hashlib.sha256(NOTEBOOK.read_bytes()).hexdigest(),
                git_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip(),
                python=sys.version, numpy=np.__version__, polars=pl.__version__)
(RUN / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(RUN / "source_provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))